# Deep Custom CNN for Traffic Density Classification

Notebook của M2 xây dựng CNN sâu hơn, huấn luyện từ đầu để phân loại ảnh thành năm mức mật độ giao thông. M1 đạt khoảng 97,12% accuracy trên train ở epoch 10, 78,45% trên validation và 76,08% trên test (macro F1: 75,82%). Khoảng cách train-validation cho thấy Simple CNN có dấu hiệu overfitting.

M2 khảo sát kiến trúc sâu hơn cùng BatchNormalization, GlobalAveragePooling2D, Dropout, augmentation và class weights để cải thiện khả năng tổng quát hóa. Mục tiêu không chỉ là tăng số lớp mà còn kiểm soát overfitting và chú ý đến các lớp ít ảnh.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    project_dir = Path("/content/dl-traffic-density-classi")

    DRIVE_PROJECT_DIR = Path(
        "/content/drive/MyDrive/dl-traffic-density-classi"
    )

    DATASET_DIR = DRIVE_PROJECT_DIR / "Final Dataset"
    OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"

    if not project_dir.exists():
        raise FileNotFoundError(
            "Repo not found. Clone it first to "
            "/content/dl-traffic-density-classi"
        )

    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f"Dataset not found: {DATASET_DIR}"
        )

    dataset_link = project_dir / "Final Dataset"

    if dataset_link.is_symlink():
        if dataset_link.resolve() != DATASET_DIR.resolve():
            dataset_link.unlink()
            dataset_link.symlink_to(
                DATASET_DIR,
                target_is_directory=True,
            )
    elif not dataset_link.exists():
        dataset_link.symlink_to(
            DATASET_DIR,
            target_is_directory=True,
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    os.chdir(project_dir)

else:
    project_dir = Path.cwd()

    if not (project_dir / "src").exists():
        project_dir = project_dir.parent

    DATASET_DIR = project_dir / "Final Dataset"
    OUTPUT_DIR = project_dir / "outputs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

print("Project root:", project_dir)
print("Dataset:", DATASET_DIR)
print("Output:", OUTPUT_DIR)

In [6]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models

In [7]:
manifest_dir = project_dir / "data" / "manifests"
required_manifests = [
    manifest_dir / "train.csv",
    manifest_dir / "val.csv",
    manifest_dir / "test.csv",
]
missing_manifests = [path for path in required_manifests if not path.exists()]
if missing_manifests:
    raise FileNotFoundError(
        "Missing manifest file(s): " + ", ".join(map(str, missing_manifests))
    )

from src.data_loader import CLASS_NAMES, create_generators
from src.evaluation import evaluate_model

print("Shared src imports: OK")
print("Manifest directory:", manifest_dir)


## 1. Thiết lập thí nghiệm

In [8]:
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)


Image size: (224, 224)
Batch size: 32
Maximum epochs: 20


Dùng cùng seed, kích thước ảnh và batch size như pipeline chung. EarlyStopping sẽ dừng sớm nếu validation loss không còn cải thiện.

## 2. Đọc dữ liệu

In [9]:
train_generator, val_generator, test_generator = create_generators(
    augmentation=False, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)
expected_class_indices = {name: index for index, name in enumerate(CLASS_NAMES)}
print("Class order:", CLASS_NAMES)
print("Class mapping:", train_generator.class_indices)
print("Training images:", train_generator.samples)
print("Validation images:", val_generator.samples)
print("Test images:", test_generator.samples)
assert train_generator.class_indices == expected_class_indices

assert train_generator.samples == 2707, train_generator.samples
assert val_generator.samples == 580, val_generator.samples
assert test_generator.samples == 581, test_generator.samples
print("Generator sample counts: OK")


Found 2707 validated image filenames belonging to 5 classes.
Found 580 validated image filenames belonging to 5 classes.
Found 581 validated image filenames belonging to 5 classes.
Class order: ['Empty', 'Low', 'Medium', 'High', 'Traffic Jam']
Class mapping: {'Empty': 0, 'Low': 1, 'Medium': 2, 'High': 3, 'Traffic Jam': 4}
Training images: 2707
Validation images: 580
Test images: 581


Các generator đọc manifest đã làm sạch của dự án. Validation và test không augmentation, đồng thời không shuffle để nhãn giữ đúng thứ tự khi đánh giá.

## 3. Phân bố lớp và class weights

In [10]:
train_manifest = pd.read_csv(project_dir / "data" / "manifests" / "train.csv")
class_to_index = {name: index for index, name in enumerate(CLASS_NAMES)}
train_counts = train_manifest["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
train_class_ids = train_manifest["class_name"].map(class_to_index).to_numpy()
class_ids = np.arange(len(CLASS_NAMES))

class_weight_values = compute_class_weight(
    class_weight="balanced", classes=class_ids, y=train_class_ids,
)
class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_ids, class_weight_values)
}
class_weight_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Number of images": train_counts.to_numpy(),
    "Class weight": [class_weights[index] for index in class_ids],
})
class_weight_table

,Class,Number of images,Class weight
0,Empty,925,0.585297
1,Low,741,0.730634
2,Medium,549,0.986157
3,High,311,1.740836
4,Traffic Jam,181,2.991160


Class weights chỉ được tính từ nhãn trong tập train. Lớp nhiều ảnh nhận trọng số nhỏ hơn; lớp ít ảnh nhận trọng số lớn hơn, nên lỗi ở lớp hiếm đóng góp nhiều hơn vào loss. Không dùng nhãn validation hoặc test để tính trọng số.

## 4. Kiến trúc Deep Custom CNN

In [11]:
def build_deep_cnn():
    model = models.Sequential([
        layers.Input(shape=(*IMAGE_SIZE, 3)),
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(len(CLASS_NAMES), activation="softmax"),
    ])
    return model

model_preview = build_deep_cnn()
model_preview.summary()
M1_PARAMETERS = 2_792_869
print(f"M1 parameters: {M1_PARAMETERS:,}")
print(f"M2 parameters: {model_preview.count_params():,}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       294,91

 Total params: 617,829 (2.36 MB)

 Trainable params: 616,421 (2.35 MB)

 Non-trainable params: 1,408 (5.50 KB)

M1 parameters: 2,792,869
M2 parameters: 617,829


Các block tích chập sâu hơn học đặc trưng từ đơn giản đến phức tạp. BatchNormalization giúp ổn định quá trình học. GlobalAveragePooling2D gộp mỗi bản đồ đặc trưng thành một giá trị, giảm mạnh số tham số ở phần phân loại so với Flatten của M1. Dropout giảm phụ thuộc quá mức vào một số neuron và hỗ trợ hạn chế overfitting. Đây là CNN tự thiết kế, không dùng pretrained weights.

## 5. Hàm hỗ trợ huấn luyện và trực quan hóa

In [12]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6,
        ),
    ]

def plot_history(history, title):
    epochs = range(1, len(history.history["loss"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["loss"], label="Training loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{title} - Loss"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["accuracy"], label="Training accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"{title} - Accuracy"); plt.legend()
    plt.tight_layout()
    plt.show()

def get_validation_macro_f1(model, generator):
    generator.reset()
    probabilities = model.predict(generator, verbose=0)
    predicted_labels = np.argmax(probabilities, axis=1)
    return f1_score(generator.classes, predicted_labels, average="macro", zero_division=0)

def record_experiment(name, history, model, val_generator, augmentation, weighted):
    best_val_accuracy = max(history.history["val_accuracy"])
    lowest_val_loss = min(history.history["val_loss"])
    val_macro_f1 = get_validation_macro_f1(model, val_generator)
    experiment_models[name] = model
    experiment_histories[name] = history
    experiment_records.append({
        "Experiment": name, "Augmentation": augmentation, "Class Weights": weighted,
        "Best Val Accuracy": best_val_accuracy, "Best Val Macro F1": val_macro_f1,
        "Lowest Val Loss": lowest_val_loss,
        "Epochs Trained": len(history.history["loss"]),
    })
    print(f"Best validation accuracy: {best_val_accuracy:.4f}")
    print(f"Validation macro F1: {val_macro_f1:.4f}")
    print(f"Lowest validation loss: {lowest_val_loss:.4f}")

Callbacks dừng huấn luyện khi validation loss không cải thiện và giảm learning rate khi cần. Macro F1 trên validation giúp so sánh công bằng hơn giữa các lớp có số lượng ảnh khác nhau.

## 6. Experiment 1 — Deep CNN baseline

Thí nghiệm này dùng kiến trúc mới nhưng không augmentation và không class weights, nhằm quan sát ảnh hưởng của kiến trúc.

In [13]:
experiment_models = {}
experiment_histories = {}
experiment_records = []

train_generator_1, val_generator_1, _ = create_generators(
    augmentation=False, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)
model_exp1 = build_deep_cnn()
model_exp1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
history_exp1 = model_exp1.fit(
    train_generator_1, validation_data=val_generator_1, epochs=EPOCHS,
    callbacks=make_callbacks(),
)
plot_history(history_exp1, "Experiment 1: Deep CNN baseline")
record_experiment(
    "Deep CNN baseline", history_exp1, model_exp1, val_generator_1,
    augmentation=False, weighted=False,
)

Found 2707 validated image filenames belonging to 5 classes.
Found 580 validated image filenames belonging to 5 classes.
Found 581 validated image filenames belonging to 5 classes.
Epoch 1/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 246s 3s/step - accuracy: 0.5364 - loss: 1.1093 - val_accuracy: 0.3414 - val_loss: 3.4433 - learning_rate: 0.0010
Epoch 2/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 257s 3s/step - accuracy: 0.6395 - loss: 0.8477 - val_accuracy: 0.3414 - val_loss: 3.5818 - learning_rate: 0.0010
Epoch 3/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 249s 3s/step - accuracy: 0.6712 - loss: 0.7839 - val_accuracy: 0.3483 - val_loss: 2.7508 - learning_rate: 0.0010
Epoch 4/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 245s 3s/step - accuracy: 0.7178 - loss: 0.6808 - val_accuracy: 0.3483 - val_loss: 2.5747 - learning_rate: 0.0010
Epoch 5/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 252s 3s/step - accuracy: 0.7425 - loss: 0.6386 - val_accuracy: 0.3448 - val_loss: 2.9911 - learning_rate: 0.0010
Epoch 6/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 247s 3s/step - accuracy: 0

KeyboardInterrupt: 

Sau khi chạy, quan sát khoảng cách giữa các đường train và validation, cũng như việc validation loss có tăng trong khi train loss tiếp tục giảm hay không. Không dùng test set ở thí nghiệm này.

## 7. Experiment 2 — Deep CNN + Augmentation

Kiến trúc và thiết lập giữ nguyên; chỉ bật augmentation. Mô hình mới được khởi tạo từ đầu.

In [ ]:
train_generator_2, val_generator_2, _ = create_generators(
    augmentation=True, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)
model_exp2 = build_deep_cnn()
model_exp2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
history_exp2 = model_exp2.fit(
    train_generator_2, validation_data=val_generator_2, epochs=EPOCHS,
    callbacks=make_callbacks(),
)
plot_history(history_exp2, "Experiment 2: Deep CNN + Augmentation")
record_experiment(
    "Deep CNN + Augmentation", history_exp2, model_exp2, val_generator_2,
    augmentation=True, weighted=False,
)

Dùng biểu đồ để kiểm tra augmentation có làm giảm khoảng cách train-validation hay không. Ghi nhận kết quả thực tế sau khi chạy notebook.

## 8. Experiment 3 — Deep CNN + Augmentation + Class Weights

Thí nghiệm này thêm class weights vào cấu hình có augmentation để xem hiệu quả trên các lớp ít ảnh. Mô hình lại được khởi tạo mới.

In [ ]:
train_generator_3, val_generator_3, test_generator = create_generators(
    augmentation=True, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)
model_exp3 = build_deep_cnn()
model_exp3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
history_exp3 = model_exp3.fit(
    train_generator_3, validation_data=val_generator_3, epochs=EPOCHS,
    callbacks=make_callbacks(), class_weight=class_weights,
)
plot_history(history_exp3, "Experiment 3: Deep CNN + Augmentation + Class Weights")
record_experiment(
    "Deep CNN + Augmentation + Class Weights", history_exp3, model_exp3, val_generator_3,
    augmentation=True, weighted=True,
)

Kiểm tra các đường cong và validation macro F1 để xem class weights có hỗ trợ các lớp hiếm mà không làm giảm đáng kể khả năng phân loại chung hay không.

## 9. So sánh các thí nghiệm và chọn cấu hình cuối

In [ ]:
experiment_summary = pd.DataFrame(experiment_records)
experiment_summary

In [ ]:
selected_experiment = max(
    experiment_records,
    key=lambda row: (
        row["Best Val Macro F1"], row["Best Val Accuracy"], -row["Lowest Val Loss"]
    ),
)
final_model = experiment_models[selected_experiment["Experiment"]]
print("Selected final configuration:", selected_experiment["Experiment"])
print("Selection uses validation macro F1, with validation accuracy and loss as tie-breakers.")

Cấu hình cuối được chọn dựa trên validation macro F1; nếu bằng nhau thì ưu tiên accuracy cao hơn rồi validation loss thấp hơn. Bảng và lựa chọn được tạo từ kết quả thực tế sau khi chạy, không dựa trên test set.

## 10. Đánh giá mô hình M2 trên test set

In [ ]:
m2_result = evaluate_model(
    model=final_model, test_generator=test_generator, model_name="Deep Custom CNN",
)
m2_result

Đây là lần duy nhất dùng test set để báo cáo metrics tổng thể cho cấu hình M2 đã chọn. Precision, Recall và F1-score dùng macro averaging.

## 11. Confusion Matrix và kết quả theo từng lớp

Confusion matrix và classification report cho thấy lớp nào thường bị nhầm lẫn. Đặc biệt chú ý High và Traffic Jam vì đây là các lớp ít mẫu hơn trong tập train. Diễn giải dựa trên kết quả thực tế sau khi chạy.

In [ ]:
test_generator.reset()
pred_probs = final_model.predict(test_generator, verbose=0)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix - Deep Custom CNN")
plt.colorbar()
tick_positions = np.arange(len(CLASS_NAMES))
plt.xticks(tick_positions, CLASS_NAMES, rotation=35, ha="right")
plt.yticks(tick_positions, CLASS_NAMES)
plt.xlabel("Predicted class")
plt.ylabel("True class")
threshold = cm.max() / 2 if cm.size else 0
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        plt.text(column, row, format(cm[row, column], "d"),
                 ha="center", va="center",
                 color="white" if cm[row, column] > threshold else "black")
plt.tight_layout()
plt.show()

print(classification_report(
    y_true, y_pred, labels=np.arange(len(CLASS_NAMES)),
    target_names=CLASS_NAMES, zero_division=0,
))

## 12. So sánh với Simple CNN của M1

In [ ]:
m1_result = pd.DataFrame({
    "Model": ["Simple CNN"],
    "Accuracy": [0.7608],
    "Precision": [0.7459],
    "Recall": [0.7752],
    "F1-score": [0.7582],
})
comparison = pd.concat([m1_result, m2_result], ignore_index=True)
comparison

In [ ]:
m1_accuracy = float(m1_result.loc[0, "Accuracy"])
m1_macro_f1 = float(m1_result.loc[0, "F1-score"])
m2_accuracy = float(m2_result.loc[0, "Accuracy"])
m2_macro_f1 = float(m2_result.loc[0, "F1-score"])

print(f"Accuracy improved over M1: {m2_accuracy > m1_accuracy}")
print(f"Macro F1 improved over M1: {m2_macro_f1 > m1_macro_f1}")
print(f"Accuracy difference (M2 - M1): {m2_accuracy - m1_accuracy:+.4f}")
print(f"Macro F1 difference (M2 - M1): {m2_macro_f1 - m1_macro_f1:+.4f}")

Dựa vào bảng để nhận xét liệu Accuracy và macro F1 có cải thiện so với M1 hay không. Kết hợp confusion matrix, per-class scores và đường cong train-validation để nhận xét khả năng giảm overfitting và mức cân bằng giữa các lớp. Chỉ ghi kết luận sau khi notebook đã chạy.

## 13. Nhận xét

Sau khi chạy các thí nghiệm, tóm tắt ngắn gọn kết quả thực tế: tác động của kiến trúc, augmentation và class weights; mức overfitting quan sát được; hiệu quả trên High và Traffic Jam; và giới hạn do quy mô dữ liệu. Không điền trước các kết luận hoặc metrics chưa được đo.

## 14. Lưu mô hình

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / "deep_custom_cnn.keras"
final_model.save(model_path)
print(f"Model saved to: {model_path}")
